# Classificação binária com Keras

Neste notebook, vamos treinar uma rede neural pequena para classificar tumores como benignos ou malignos usando o conjunto Breast Cancer Wisconsin, incluído no scikit-learn. O exemplo usa Keras com TensorFlow como backend e não precisa baixar dados externos.

> Se o TensorFlow ainda não estiver instalado no ambiente do projeto, execute no terminal, a partir da raiz: `uv add tensorflow`. Em seguida, escolha no notebook o kernel Python do `.venv` desse projeto.

## 1. Importações e reprodutibilidade

Definimos uma semente para tornar a divisão dos dados e a inicialização do treinamento reproduzíveis.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

keras.utils.set_random_seed(42)

print(f"TensorFlow: {tf.__version__}")

## 2. Carregar e observar os dados

Cada observação tem 30 medidas numéricas. No dataset original, o alvo `0` representa tumor maligno e o alvo `1` representa tumor benigno. Vamos exibir as quantidades e algumas features para entender a entrada do modelo.

In [ ]:
dataset = load_breast_cancer()
X = dataset.data.astype("float32")
y = dataset.target.astype("float32")

print(f"Formato das features: {X.shape}")
print(f"Formato do alvo: {y.shape}")
print("Classes:", dict(enumerate(dataset.target_names)))
print("Contagem por classe:", dict(zip(*np.unique(y.astype(int), return_counts=True))))

## 3. Separar treino e teste e padronizar

Reservamos 20% das observações para o teste final. `stratify=y` mantém aproximadamente a mesma proporção de classes nos dois grupos. O `StandardScaler` é ajustado **somente com os dados de treino**; assim, as estatísticas do conjunto de teste não vazam para o treinamento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print(f"Treino: {X_train_scaled.shape}; teste: {X_test_scaled.shape}")

## 4. Criar e treinar a rede neural

A camada de saída tem um neurônio com ativação sigmoide, que retorna uma probabilidade para a classe `1` (benigno). Usamos entropia cruzada binária como função de perda. O `EarlyStopping` encerra o treino se a perda de validação não melhorar e restaura os melhores pesos encontrados.

In [ ]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="classificador_tumor",
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0,
)

print(f"Épocas executadas: {len(history.history['loss'])}")
print("Melhor acurácia de validação:", max(history.history["val_accuracy"]))

## 5. Avaliar no conjunto de teste

O conjunto de teste não participou do ajuste do scaler nem do treinamento. A matriz de confusão cruza os rótulos reais (linhas) com as previsões (colunas), na ordem maligno (`0`) e benigno (`1`).

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
probabilities = model.predict(X_test_scaled, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f"Perda no teste: {test_loss:.4f}")
print(f"Acurácia no teste: {test_accuracy:.4f}")
print("\nMatriz de confusão (linhas=reais, colunas=previstas):")
print(confusion_matrix(y_test.astype(int), predictions, labels=[0, 1]))
print("\nRelatório de classificação:")
print(
    classification_report(
        y_test.astype(int),
        predictions,
        labels=[0, 1],
        target_names=dataset.target_names,
        zero_division=0,
    )
)

## 6. Fazer uma previsão individual

A rede devolve a probabilidade da classe `1` (benigno). Aplicamos o limiar de 0,5 para converter essa probabilidade em uma classe prevista.

In [ ]:
sample = X_test_scaled[:1]
benign_probability = float(model.predict(sample, verbose=0)[0, 0])
predicted_class = int(benign_probability >= 0.5)

print(f"Probabilidade de benigno (classe 1): {benign_probability:.3f}")
print(f"Classe prevista: {predicted_class} — {dataset.target_names[predicted_class]}")
print(f"Classe real: {int(y_test[0])} — {dataset.target_names[int(y_test[0])]}")

## Conclusão

Este exemplo percorreu o fluxo de uma classificação com rede neural: preparação dos dados, treino com validação e avaliação em observações separadas. A acurácia, sozinha, não resume o desempenho de um classificador; por isso também analisamos precisão, recall, F1-score e a matriz de confusão. Em um cenário médico real, seriam necessários validação clínica, análise de erros e controles adicionais antes de qualquer uso prático.